# Q-BioDiag: Quantum AI for Biomedical Diagnostics

Hybrid Qiskit QSVC + Pinecone vector database demo for early lesion detection.

## 1. Generate synthetic biomedical image patches
The demo uses reproducible 16x16 microscopy-style patches: benign texture vs early lesion microcalcification clusters.

In [ ]:
from qbio_diagnostics.data import generate_synthetic_biomedical_patches, save_montage
dataset = generate_synthetic_biomedical_patches(n_samples=160)
save_montage(dataset, '../outputs/sample_patches.png')
dataset.images.shape, dataset.labels.mean()

## 2. Compress images to qubit features
We compress diagnostic image descriptors to four real-valued features mapped to four qubits.

In [ ]:
from qbio_diagnostics.features import make_qubit_features
bundle = make_qubit_features(dataset.images, dataset.labels, qubits=4)
bundle.x_train.shape, bundle.x_test.shape

## 3. Classical baseline vs Quantum QSVC
QSVC uses Qiskit's ZZFeatureMap + FidelityQuantumKernel. If Qiskit is not installed, the script keeps running with a polynomial SVM fallback.

In [ ]:
from qbio_diagnostics.train_classical import train_classical_svm
from qbio_diagnostics.train_qsvc import train_quantum_qsvc
classical_model, c_pred, c_score, c_metrics = train_classical_svm(bundle.x_train, bundle.y_train, bundle.x_test, bundle.y_test)
q_model, q_pred, q_score, q_metrics, backend = train_quantum_qsvc(bundle.x_train, bundle.y_train, bundle.x_test, bundle.y_test)
c_metrics, q_metrics, backend

## 4. Save figures for pitch deck/demo

In [ ]:
from qbio_diagnostics.metrics import save_confusion_matrix, save_roc_curve
save_confusion_matrix(bundle.y_test, q_pred, '../outputs/quantum_confusion_matrix.png', 'Q-BioDiag QSVC Confusion Matrix')
save_roc_curve(bundle.y_test, q_score, '../outputs/quantum_roc_curve.png', 'Q-BioDiag QSVC ROC Curve')

## 5. Pinecone vector DB integration
The same compressed vectors can be stored in Pinecone for similar-case retrieval. Without an API key, a local JSON vector store is created.

In [ ]:
from qbio_diagnostics.pinecone_store import upsert_cases
upsert_cases(bundle.x_train, bundle.y_train, out_dir='../outputs', use_pinecone=False)